In [1]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
# 
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    classification_report
)

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce MX450


In [3]:
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/validation.csv")
test_df = pd.read_csv("../data/processed/test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (4135, 3)
Validation: (517, 3)
Test: (517, 3)


In [5]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={
        0: "ham",
        1: "spam"
    },
    label2id={
        "ham": 0,
        "spam": 1
    }
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
MAX_LENGTH = 128

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [7]:
train_dataset = Dataset.from_pandas(
    train_df[["text", "label_id"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["text", "label_id"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label_id"]],
    preserve_index=False
)

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

train_tokenized = train_tokenized.rename_column(
    "label_id",
    "labels"
)

val_tokenized = val_tokenized.rename_column(
    "label_id",
    "labels"
)

test_tokenized = test_tokenized.rename_column(
    "label_id",
    "labels"
)

Map:   0%|          | 0/4135 [00:00<?, ? examples/s]

Map:   0%|          | 0/517 [00:00<?, ? examples/s]

Map:   0%|          | 0/517 [00:00<?, ? examples/s]

In [8]:
r=32
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=32,
    lora_alpha=2*r,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    bias="none"
)

In [9]:
model = get_peft_model(
    model,
    lora_config
)

In [10]:
model.print_trainable_parameters()

trainable params: 1,181,954 || all params: 68,136,964 || trainable%: 1.7347


In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "spam_f1": f1,
        "macro_f1": macro_f1
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="../results/lora_r32",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    learning_rate=2e-5,
    weight_decay=0.01,

    num_train_epochs=3,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    logging_steps=25,

    seed=42,

    report_to="none",

    fp16=True
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [14]:
trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Spam F1,Macro F1
1,0.069720,0.063719,0.980658,0.937500,0.909091,0.923077,0.956007
2,0.055835,0.051764,0.982592,0.938462,0.924242,0.931298,0.960665
3,0.030247,0.050994,0.984526,0.953125,0.924242,0.938462,0.964806


TrainOutput(global_step=777, training_loss=0.10754091701586581, metrics={'train_runtime': 674.1188, 'train_samples_per_second': 18.402, 'train_steps_per_second': 1.153, 'total_flos': 200217701832192.0, 'train_loss': 0.10754091701586581, 'epoch': 3.0})

In [15]:
test_results = trainer.evaluate(
    test_tokenized
)

test_results

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,Spam F1,Macro F1
0.030247,0.032848,3,0.994197,1.000000,0.953846,0.976378,0.986535


{'eval_loss': 0.032848238945007324,
 'eval_accuracy': 0.9941972920696325,
 'eval_precision': 1.0,
 'eval_recall': 0.9538461538461539,
 'eval_spam_f1': 0.9763779527559056,
 'eval_macro_f1': 0.986535172629331}

In [16]:
print(model.active_adapter)
model.print_trainable_parameters()


default
trainable params: 1,181,954 || all params: 68,136,964 || trainable%: 1.7347


In [18]:
model.save_pretrained("models/distilbert-sms-spam-lora-r32")
tokenizer.save_pretrained("models/distilbert-sms-spam-lora-r32")

('models/distilbert-sms-spam-lora-r32/tokenizer_config.json',
 'models/distilbert-sms-spam-lora-r32/tokenizer.json')